# Checkpoint 3: Running the Statistical Tests

t-test (2 groups), ANOVA + Bonferroni (5 groups), and also chi-square (two categorical variables) just to show the full range.

## 1. Loading df_clean

In [155]:
import pandas as pd
import numpy as np
from scipy import stats
from itertools import combinations

pd.set_option('display.max_columns', 40)

In [156]:
cols = [
    "id_x", "car_rel_url_x", "datetime_scrape", "price_x", "currency_x", "city",
    "production_year", "engine_displacement_num", "kilometrage_num", "Marka", "Model",
    "Sürətlər qutusu", "Vəziyyəti", "Ötürücü", "Ban növü", "views"
]

In [157]:
df = pd.read_csv("cars.csv", usecols=cols, parse_dates=["datetime_scrape"])

In [158]:
df

,id_x,car_rel_url_x,datetime_scrape,price_x,currency_x,city,production_year,engine_displacement_num,kilometrage_num,views,Ban növü,Marka,Model,Sürətlər qutusu,Vəziyyəti,Ötürücü
0,3c234145-d57a-4ad6-9448-d43810fc3392,/autos/8748840-hyundai-i30,2024-09-13 20:32:19.751157+00,15000.0,AZN,bakı,2008,1.6,270000,492,"Hetçbek, 5 qapı",Hyundai,i30,Mexaniki,"Vuruğu yoxdur, rənglənməyib",Ön
1,c74ea36f-6be1-4de4-926d-e117197dcf00,/autos/8475807-lada-vaz-niva-travel,2024-09-13 20:32:19.751157+00,23700.0,AZN,bakı,2024,1.7,0,60189,"Offroader / SUV, 5 qapı",LADA (VAZ),Niva Travel,Mexaniki,"Vuruğu yoxdur, rənglənməyib",Tam
2,9cefceb0-024d-4581-a869-a3c2c68a9f95,/autos/8739686-toyota-land-cruiser,2024-09-13 20:32:19.751157+00,35600.0,$,bakı,2011,4.0,164750,2473,"Offroader / SUV, 5 qapı",Toyota,Land Cruiser,Avtomat,"Vuruğu yoxdur, rənglənməyib",Tam
3,459cc337-fb63-48de-9694-41554923d311,/autos/8712597-hyundai-elantra,2024-09-13 20:32:19.751157+00,26700.0,AZN,bakı,2018,2.0,126000,3727,Sedan,Hyundai,Elantra,Avtomat,"Vuruğu yoxdur, rənglənməyib",Ön
4,6c5ee8d8-1c6f-4fad-a694-957a4c43c25d,/autos/8674773-toyota-prius,2024-09-13 20:32:19.751157+00,10500.0,AZN,bakı,2007,1.5,354000,446,Liftbek,Toyota,Prius,Variator,"Vuruğu yoxdur, rənglənməyib",Ön
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
653716,16caa803-a546-455b-81ff-bc20868c2136,/autos/9081944-toyota-prius,2025-01-05 20:15:21.051803,10800.0,AZN,bakı,2008,1.5,320000,210,Liftbek,Toyota,Prius,Variator,"Vuruğu yoxdur, rənglənməyib",Ön
653717,2a26a6c1-8914-4b68-abb4-1fbd12ced7bc,/autos/9081939-uaz-hunter,2025-01-05 20:15:21.051803,9500.0,AZN,göygöl,2011,2.9,155000,1195,"Offroader / SUV, 5 qapı",UAZ,Hunter,Mexaniki,"Vuruğu yoxdur, rənglənməyib",Tam
653718,feb75615-0131-4d6f-b009-cc02faea4e01,/autos/9055034-hyundai-elantra,2025-01-05 20:15:21.051803,25400.0,AZN,bakı,2018,2.0,77926,1120,Sedan,Hyundai,Elantra,Avtomat,"Vuruğu yoxdur, rənglənməyib",Ön
653719,e5f8e957-5543-42be-b30f-78723ce551f9,/autos/9065903-jeep-grand-cherokee,2025-01-05 20:15:21.051803,10600.0,AZN,kürdəmir,1999,4.7,250000,1320,"Offroader / SUV, 5 qapı",Jeep,Grand Cherokee,Avtomat,"Vuruğu yoxdur, rənglənməyib",Tam


In [159]:
df_dedup = df.sort_values("datetime_scrape").drop_duplicates(subset = "car_rel_url_x", keep = "last").copy()

In [160]:
exchange_rate = {"AZN": 1.0, "$": 1.70, "€": 1.85}
df_dedup["price_azn"] = df_dedup["price_x"] * df_dedup["currency_x"].map(exchange_rate)

In [161]:
exclude_body_types = ["Yük maşını", "Motosiklet", "Avtobus", "Moped", "Kvadrosikl", "Dartqı", "Mikroavtobus"]
df_clean = df_dedup[~df_dedup["Ban növü"].isin(exclude_body_types)].copy()
df_clean = df_clean[df_clean["price_azn"] >= 1000].copy()

In [162]:
df_clean.shape

(149478, 17)

## 2. Test 1: Independent t-test (gearbox vs price)

Since the sample is huge (37,000+ per group), I use Welch's t-test since it doesn't assume equal variances

In [163]:
manual = df_clean.loc[df_clean["Sürətlər qutusu"] == "Mexaniki", "price_azn"]
auto = df_clean.loc[df_clean["Sürətlər qutusu"] == "Avtomat", "price_azn"]


In [164]:
print(f"Manual: n={len(manual)}, mean={manual.mean():.0f}, std={manual.std():.0f}")
print(f"Automatic: n={len(auto)}, mean={auto.mean():.0f}, std={auto.std():.0f}")

Manual: n=37161, mean=10567, std=8188
Automatic: n=98161, mean=30023, std=34053


In [165]:
t_stat, p_value = stats.ttest_ind(auto, manual, equal_var=False)  # Welch's t-test

In [166]:
# Cohen's d (effect size)
pooled_std = np.sqrt(((len(auto) - 1) * auto.var() + (len(manual) - 1) * manual.var()) / (len(auto) + len(manual) - 2))
cohens_d = (auto.mean() - manual.mean()) / pooled_std

In [167]:
# 95% CI using a normal approximation (sample is huge, so z=1.96 works fine)
diff = auto.mean() - manual.mean()
se = np.sqrt(auto.var() / len(auto) + manual.var() / len(manual))
ci_low, ci_high = diff - 1.96 * se, diff + 1.96 * se

In [168]:
print(f"Welch t-statistic: {t_stat:.2f}")
print(f"p-value: {p_value:.2e}")
print(f"Mean price difference (Automatic - Manual): {diff:.0f} AZN")
print(f"95% CI: [{ci_low:.0f}, {ci_high:.0f}] AZN")
print(f"Cohen's d: {cohens_d:.3f}")

Welch t-statistic: 166.72
p-value: 0.00e+00
Mean price difference (Automatic - Manual): 19456 AZN
95% CI: [19227, 19684] AZN
Cohen's d: 0.664


the p-value is basically 0, and the 95% CI [19227, 19684] AZN doesn't include zero, so there's a significant difference. Cohen's d 0.66, a medium-to-large effect size.

## 3. One-way ANOVA + Bonferroni (brand vs price)

Continuous dependent variable + categorical independent variable with 5 groups, one-way ANOVA.

In [169]:
top5_brands = ["Mercedes", "Hyundai", "Kia", "Toyota", "LADA (VAZ)"]
groups = [df_clean.loc[df_clean["Marka"] == b, "price_azn"].values for b in top5_brands]


In [170]:
for b, g in zip(top5_brands, groups):
    print(f"{b}: n={len(g)}, mean={g.mean():.0f}")

Mercedes: n=25307, mean=25650
Hyundai: n=19658, mean=23864
Kia: n=14619, mean=25765
Toyota: n=14388, mean=30287
LADA (VAZ): n=12859, mean=7264


In [171]:
F_stat, p_value_anova = stats.f_oneway(*groups)

In [172]:
# eta-squared (effect size)
grand_mean = np.concatenate(groups).mean()
ss_between = sum(len(g) * (g.mean() - grand_mean) ** 2 for g in groups)
ss_total = sum(((g - grand_mean) ** 2).sum() for g in groups)
eta_squared = ss_between / ss_total

In [173]:
print(f"ANOVA F-statistic: {F_stat:.2f}")
print(f"p-value: {p_value_anova:.2e}")
print(f"Eta-squared: {eta_squared:.3f}")

ANOVA F-statistic: 1942.50
p-value: 0.00e+00
Eta-squared: 0.082


p = 0, I reject H0. But ANOVA only tells me "at least one is different". To find out which pairs, I run Bonferroni-corrected tests next.

In [174]:
alpha = 0.05
n_comparisons = len(list(combinations(top5_brands, 2)))
alpha_corrected = alpha / n_comparisons
print(f"Number of pairs: {n_comparisons}, Bonferroni-corrected α: {alpha_corrected:.4f}")
print()

Number of pairs: 10, Bonferroni-corrected α: 0.0050



In [175]:
posthoc_results = []
for b1, b2 in combinations(top5_brands, 2):
    g1 = df_clean.loc[df_clean["Marka"] == b1, "price_azn"]
    g2 = df_clean.loc[df_clean["Marka"] == b2, "price_azn"]
    t_stat, p_raw = stats.ttest_ind(g1, g2, equal_var=False)
    posthoc_results.append({
        "Pair": f"{b1} vs {b2}",
        "Mean diff": round(g1.mean() - g2.mean(), 0),
        "p-value (raw)": p_raw,
        "p < 0.05 (uncorrected)": p_raw < alpha,
        "p < corrected α (Bonferroni)": p_raw < alpha_corrected,
    })

In [176]:
posthoc_df = pd.DataFrame(posthoc_results)
posthoc_df

,Pair,Mean diff,p-value (raw),p < 0.05 (uncorrected),p < corrected α (Bonferroni)
0,Mercedes vs Hyundai,1786.0,4.516971e-13,True,True
1,Mercedes vs Kia,-115.0,6.527356e-01,False,False
2,Mercedes vs Toyota,-4638.0,9.513534e-53,True,True
3,Mercedes vs LADA (VAZ),18386.0,0.000000e+00,True,True
4,Hyundai vs Kia,-1901.0,1.361007e-56,True,True
5,Hyundai vs Toyota,-6424.0,3.022574e-216,True,True
6,Hyundai vs LADA (VAZ),16600.0,0.000000e+00,True,True
7,Kia vs Toyota,-4523.0,3.173328e-99,True,True
8,Kia vs LADA (VAZ),18501.0,0.000000e+00,True,True
9,Toyota vs LADA (VAZ),23024.0,0.000000e+00,True,True


9 out of 10 brand pairs are significant, both before and after Bonferroni correction. Only Mercedes vs Kia isn't significant (p=0.65), so I can't tell those two brands' prices apart statistically, but every other pair is different.

## 4. Chi-square (gearbox vs drivetrain)

This goes beyond the 2 main questions but shows the full method. Is there a link between gearbox and drivetrain? Both are categorical, so I use a chi-square test of independence instead of a t-test/ANOVA.

In [177]:
contingency_table = pd.crosstab(df_clean["Sürətlər qutusu"], df_clean["Ötürücü"])
contingency_table


Ötürücü,Arxa,Tam,Ön
Sürətlər qutusu,,,
Avtomat,27499,24826,45836
Mexaniki,13236,3037,20888
Reduktor,359,897,588
Robot,89,180,1258
Variator,112,671,10002


In [178]:
chi2_stat, p_chi2, dof_chi2, expected_freq = stats.chi2_contingency(contingency_table)

# Cramer's V (effect size)
n_total = contingency_table.sum().sum()
cramers_v = np.sqrt(chi2_stat / (n_total * (min(contingency_table.shape) - 1)))

In [179]:
print(f"Chi-square statistic: {chi2_stat:.1f}")
print(f"Degrees of freedom: {dof_chi2}")
print(f"p-value: {p_chi2:.2e}")
print(f"Cramer's V: {cramers_v:.3f}")
print(f"Minimum expected frequency: {expected_freq.min():.0f} (should be ≥5)")


Chi-square statistic: 14246.6
Degrees of freedom: 8
p-value: 0.00e+00
Cramer's V: 0.218
Minimum expected frequency: 302 (should be ≥5)


p = 0, and the smallest expected count (302) is well above the minimum needed for chi-square (>=5). Cramer's V = 0.22. So gearbox type and drivetrain type are statistically linked, but only moderately.